# Module 1 Bridge -- combined FIRE + LongDR + Tianjin caching

Replaces notebooks 04 and 05's caching cells with a single session: one Drive mount, one
GitHub token, one repo clone, no manual file upload needed on this branch. Builds
all 3 Module 1 output caches (FIRE, LongDR, Tianjin) that notebook 06 needs for the real
combined Module 2 training run.

**Deliberately does NOT run notebook 04's own standalone Module 2 training cell** -- that was
a throwaway 5-epoch FIRE+LongDR-only POC that nothing downstream actually reads (notebook 06
trains its own Module 2 checkpoint from scratch using all 3 sources, and notebook 08 reads
06's checkpoint, not 04's). Skipping it saves a full training run for zero loss.

**After this notebook finishes:** go straight to notebook 06, then 07 and 08.

**`MODULE1_SOURCE`** (first code cell) picks the Module 1 models: `'drgnet_combined'` (default) = notebook 11's FGADR classifier + FGADR + DDR combined lesion segmenters (caches go to `module1_cache_drgnet_comb/`); `'drgnet'` = notebook 11's FGADR classifier + FGADR-only segmenters (`module1_cache_drgnet/`); `'poc'` = notebooks 02/03's (`module1_cache/`). Each setting has its own folders, so nothing is overwritten when you switch.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
# Free Drive space: the training notebooks save checkpoints and caches here (several GB in total).
# Deleted or overwritten files stay in the Drive Trash and still count -- empty it if space is low.
import shutil as _shutil
try:
    _du = _shutil.disk_usage('/content/drive/MyDrive')
    print(f'Google Drive: {_du.free / 1e9:.1f} GB free of {_du.total / 1e9:.1f} GB')
    if _du.free < 10e9:
        print('⚠ Under 10 GB free on Google Drive -- model files may fail to save. Empty the Trash '
              '(drive.google.com/drive/trash) or free space before training.')
except OSError:
    pass

import getpass, os

GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'AllModuleEdits-PreFinals'  # <-- pre-finals fixes (CODE_FIXES_AllModuleEdits-PreFinals.md); 'mod-2-NonLongIntegration' = the earlier runs
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'  # <-- change if you used a different folder
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"

# Which Module 1 models the pipeline uses -- set the SAME value in notebooks 04_05 and 06-10:
#   'drgnet' -- notebook 11's DRG-Net replication checkpoints (module1_runs/drgnet_replication/)
#   'poc'    -- notebooks 02/03's original POC checkpoints (module1_runs/)
# Everything built from Module 1's outputs (Module 1 caches, Module 2 runs, figures) gets a
# '_drgnet' suffix for 'drgnet', so the two pipelines never overwrite each other's results.
MODULE1_SOURCE = 'drgnet_combined'
assert MODULE1_SOURCE in ('drgnet_combined', 'drgnet', 'poc'), MODULE1_SOURCE
# New Drive folders for this branch's runs: every Module 1 cache / Module 2 / Module 3 folder below
# gets RUN_SUFFIX, so the mod-2-NonLongIntegration results stay untouched for before/after
# comparison. '' = read/write the earlier folders again.
RUN_SUFFIX = '_prefinals'
RUN_TAG = {'drgnet_combined': '_drgnet_comb', 'drgnet': '_drgnet', 'poc': ''}[MODULE1_SOURCE] + RUN_SUFFIX
MODULE1_RUNS = os.path.join(DRIVE_DATA_DIR, 'module1_runs', *(['drgnet_replication'] if MODULE1_SOURCE != 'poc' else []))
# Notebook 11 on this branch writes its models to drgnet_replication<RUN_SUFFIX>/ (tiled evaluation,
# 200 segmentation epochs, optional EyePACS grader); use those once they exist.
if MODULE1_SOURCE != 'poc' and os.path.isdir(os.path.join(MODULE1_RUNS + RUN_SUFFIX, 'segmentation')):
    MODULE1_RUNS += RUN_SUFFIX
elif MODULE1_SOURCE != 'poc' and RUN_SUFFIX:
    print(f'\u26a0 {MODULE1_RUNS + RUN_SUFFIX} not found -- using the earlier Module 1 models in {MODULE1_RUNS}. '
          'Run notebook 11 on this branch first for the re-trained ones.')
# Lesion segmenters: <MODULE1_RUNS>/segmentation/<SEG_MODEL_PREFIX><lesion>/model_2.pth.tar --
# FGADR + DDR combined (notebook 11) or FGADR only. The classifier is FGADR's either way.
SEG_MODEL_PREFIX = 'comb_models_' if MODULE1_SOURCE == 'drgnet_combined' else 'models_FGADR_NO_TATL_'
MODULE1_CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_TAG)
os.makedirs(MODULE1_CACHE_DIR, exist_ok=True)
print('Module 1 source:', MODULE1_SOURCE, '| models:', MODULE1_RUNS, '| caches:', MODULE1_CACHE_DIR)
TIANJIN_ZIP = os.path.join(DRIVE_DATA_DIR, 'retinal-dr-longitudinal.zip')
assert os.path.isfile(TIANJIN_ZIP), f"Not found: {TIANJIN_ZIP}"


Mounted at /content/drive
Google Drive: 28.3 GB free of 107.4 GB
GitHub Personal Access Token (repo scope): ··········
Module 1 source: drgnet_combined | models: /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals | caches: /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals


In [2]:
# Unzip FIRE + LongDR + Tianjin. -n skips files that already exist, so safe/cheap to re-run.
import glob, shutil

os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/FIRE_dataset.zip"
!unzip -q -n "$DRIVE_DATA_DIR/LongDRScreening_20150209.zip" -d LongDRScreening_20150209
!unzip -q -n "$TIANJIN_ZIP" -d _tianjin_extract_raw

_manifest_candidates = glob.glob('/content/data/_tianjin_extract_raw/**/corrected_manifest.csv', recursive=True)
assert _manifest_candidates, 'No corrected_manifest.csv found under the extracted Tianjin zip.'
_tianjin_root = os.path.dirname(_manifest_candidates[0])
TIANJIN_DIR = '/content/data/retinal-dr-longitudinal'
if _tianjin_root != TIANJIN_DIR and not os.path.exists(TIANJIN_DIR):
    os.symlink(_tianjin_root, TIANJIN_DIR)

print('FIRE Images present :', os.path.isdir('/content/data/FIRE_dataset/FIRE/Images'))
print('LongDR norm present :', os.path.isdir('/content/data/LongDRScreening_20150209/FundusImagesNormalized'))
print('Tianjin dir present :', os.path.isdir(TIANJIN_DIR))


/content/data
FIRE Images present : True
LongDR norm present : True
Tianjin dir present : True


In [3]:
# Clone the repo once and install everything both the FIRE/LongDR and Tianjin steps need.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}

REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR}"

%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch opencv-python-headless


/content
Cloning into 'DRProgression'...
remote: Enumerating objects: 883, done.
remote: Counting objects: 100% (562/562), done.
remote: Compressing objects: 100% (368/368), done.
remote: Total 883 (delta 341), reused 388 (delta 191), pack-reused 321 (from 1)
Receiving objects: 100% (883/883), 18.56 MiB | 24.65 MiB/s, done.
Resolving deltas: 100% (511/511), done.
/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc


In [4]:
# Item 7.3: image-size distribution of the three longitudinal datasets (LongDR's and Tianjin's sizes
# were undocumented) -- for the methods section. Header-only reads, a few seconds. Saved to Drive.
%cd {REPO_CODE_DIR}
!python tools/dataset_image_sizes.py fire=/content/data/FIRE_dataset longdr=/content/data/LongDRScreening_20150209 tianjin="{TIANJIN_DIR}" --out "{MODULE1_CACHE_DIR}/dataset_image_sizes.json"

/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc

fire (/content/data/FIRE_dataset): 268 images
  most common: 2912x2912 (268)
  width  min/median/max: 2912 / 2912 / 2912
  height min/median/max: 2912 / 2912 / 2912
  non-square: 0.0%

longdr (/content/data/LongDRScreening_20150209): 2520 images
  most common: 2000x1312 (2240), 2324x1810 (2), 2268x1771 (1), 2254x1786 (1), 2415x1793 (1)
  width  min/median/max: 1622 / 2000 / 2859
  height min/median/max: 1200 / 1312 / 2650
  non-square: 100.0%

tianjin (/content/data/retinal-dr-longitudinal): 2412 images
  most common: 2992x2000 (1233), 3472x2320 (1146), 2576x1934 (32), 2144x1424 (1)
  width  min/median/max: 2144 / 2992 / 3472
  height min/median/max: 1424 / 2000 / 2320
  non-square: 100.0%

Wrote /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals/dataset_image_sizes.json


In [5]:
# Laterality resolution -- Tianjin-only, required before tianjin_dataset.py/apply_to_progression_data.py
# can run over it. Reuses a Drive-persisted copy if one exists (saves a few minutes).
# This branch re-tuned module1/resolve_eye_laterality.py (item 7.1), so it gets its own copy
# (module1_cache<RUN_SUFFIX>/); the settings come from notebook 11's EyePACS validation when it exists.
LATERALITY_DRIVE_PATH = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_SUFFIX, 'laterality_resolved.csv')
LATERALITY_VALIDATION = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_SUFFIX, 'laterality_eyepacs_validation.json')
LATERALITY_LOCAL_PATH = os.path.join(TIANJIN_DIR, 'laterality_resolved.csv')

if os.path.isfile(LATERALITY_DRIVE_PATH):
    shutil.copy(LATERALITY_DRIVE_PATH, LATERALITY_LOCAL_PATH)
    print(f'✓ Reused laterality_resolved.csv from Drive')
else:
    %cd {REPO_CODE_DIR}
    _lat_args = ''
    if os.path.isfile(LATERALITY_VALIDATION):
        import json as _json
        _b = _json.load(open(LATERALITY_VALIDATION))['best']
        _lat_args = (f" --disc-left-is {_b['disc_left_is']} --margin {_b['margin']}"
                     + (' --legacy-estimator' if _b['estimator'] == 'legacy' else ''))
        print(f"Laterality settings from the EyePACS validation: {_lat_args} "
              f"(accuracy {_b['accuracy_on_decided']:.1%}, {_b['uncertain_rate']:.1%} uncertain)")
    else:
        print(f'\u26a0 No {LATERALITY_VALIDATION} -- run notebook 11 with the EyePACS zip first to validate '
              'the optic-disc heuristic; using its default settings.')
    !python module1/resolve_eye_laterality.py --dataset-dir "{TIANJIN_DIR}"{_lat_args}
    os.makedirs(os.path.dirname(LATERALITY_DRIVE_PATH), exist_ok=True)
    shutil.copy(LATERALITY_LOCAL_PATH, LATERALITY_DRIVE_PATH)
    print(f'✓ Computed and persisted laterality_resolved.csv to Drive')


/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc
Laterality settings from the EyePACS validation:  --disc-left-is OS --margin 0.0 (accuracy 60.6%, 0.1% uncertain)
Resolved 1115/1115 eyes from their own disc position (0 uncertain -- see fallback policy in tianjin_dataset.py)
Patients with 2 baseline images resolving to the SAME eye (should be rare -- flags heuristic problems): 112/543 (20.6%)
⚠ Same-eye disagreement rate is above 5% -- try a larger --margin or smaller --downscale (or validate on EyePACS: --validate-eyepacs-dir) and re-run before using this output for training.
Pair constraint: 112 rows set to the opposite of their patient's other eye -> 0 uncertain left
Wrote /content/data/retinal-dr-longitudinal/laterality_resolved.csv
✓ Computed and persisted laterality_resolved.csv to Drive


In [6]:
# Locate Module 1 checkpoints (classifier + EX/MA + HE/SE if trained) -- shared by all 3
# apply_to_progression_data.py calls below.
import glob

SEG_DIR   = os.path.join(MODULE1_RUNS, 'segmentation')
# Module 1 grader: 'fgadr' (default) = the FGADR grader from notebook 11; 'eyepacs' = DRG-Net's classifier
# trained on EyePACS (notebook 11 RUN['eyepacs_cls'] / notebook 09 -- not trained for now). With 'eyepacs'
# and no EyePACS grader on Drive, the FGADR one is used.
CLASSIFIER_RUN = 'fgadr'
if not glob.glob(os.path.join(MODULE1_RUNS, 'classification', CLASSIFIER_RUN, 'saves', '*.pt')):
    print(f'\u26a0 No {CLASSIFIER_RUN} grader in {MODULE1_RUNS} yet -- using the FGADR one')
    CLASSIFIER_RUN = 'fgadr'
CLS_SAVES = os.path.join(MODULE1_RUNS, 'classification', CLASSIFIER_RUN, 'saves')
# Classifier input normalisation must match its training config (eyepacs_poc.yaml differs).
CLS_NORM_ARGS = ' --classifier-norm eyepacs' if CLASSIFIER_RUN == 'eyepacs' else ''

_pref  = ['final_weights.pt', 'best_validation_weights.pt']
_cands = [os.path.join(CLS_SAVES, n) for n in _pref] + sorted(
    glob.glob(os.path.join(CLS_SAVES, '*.pt')), key=os.path.getmtime, reverse=True)
CLS_CKPT = next((p for p in _cands if os.path.isfile(p)), None)
assert CLS_CKPT, f"No classifier .pt in {CLS_SAVES} -- run notebook 02/03 (MODULE1_SOURCE='poc') or 11 ('drgnet' / 'drgnet_combined') first"

EX_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'ex', 'model_2.pth.tar')
MA_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'ma', 'model_2.pth.tar')
for _p in (CLS_CKPT, EX_CKPT, MA_CKPT):
    assert os.path.isfile(_p), f"missing checkpoint: {_p} -- run notebook 02/03 (MODULE1_SOURCE='poc') or 11 ('drgnet' / 'drgnet_combined') first"

HE_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'he', 'model_2.pth.tar')
SE_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'se', 'model_2.pth.tar')
print('classifier :', CLS_CKPT)
print('EX seg     :', EX_CKPT)
print('MA seg     :', MA_CKPT)
print('HE seg     :', HE_CKPT if os.path.isfile(HE_CKPT) else 'not trained yet')
print('SE seg     :', SE_CKPT if os.path.isfile(SE_CKPT) else 'not trained yet')

_extra_seg_args = ''
if os.path.isfile(HE_CKPT):
    _extra_seg_args += f' --seg-checkpoint HE="{HE_CKPT}"'
if os.path.isfile(SE_CKPT):
    _extra_seg_args += f' --seg-checkpoint SE="{SE_CKPT}"'

os.makedirs(MODULE1_CACHE_DIR, exist_ok=True)


classifier : /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/classification/fgadr/saves/final_weights.pt
EX seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/segmentation/comb_models_ex/model_2.pth.tar
MA seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/segmentation/comb_models_ma/model_2.pth.tar
HE seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/segmentation/comb_models_he/model_2.pth.tar
SE seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/segmentation/comb_models_se/model_2.pth.tar


In [7]:
# This branch's module1/apply_to_progression_data.py already has --checkpoint-every/--resume
# (so a long run survives a lost Colab session), plus the per-lesion biomarker stats and options
# notebooks 09-11 need. The manual upload of a local copy is therefore OFF by default: an older
# local copy would replace this branch's version for this session and drop those features.
# Only set this to True if your local copy has fixes that were never pushed.
UPLOAD_LOCAL_COPY = False

if UPLOAD_LOCAL_COPY:
    from google.colab import files
    uploaded = files.upload()  # pick module1/apply_to_progression_data.py
    for fname in uploaded:
        if fname != 'apply_to_progression_data.py':
            print(f'Skipping unexpected file: {fname}')
            continue
        dest = os.path.join(REPO_CODE_DIR, 'module1', fname)
        shutil.move(fname, dest)
        print(f'Overwrote {dest}')

!python {REPO_CODE_DIR}/module1/apply_to_progression_data.py --help | grep -E "checkpoint-every|resume"


                                    [--image-list IMAGE_LIST] [--resume]
  --resume              If --out already exists, load it and skip images
  --save-every, --checkpoint-every SAVE_EVERY
                        = only at the end). --checkpoint-every is the same


In [8]:
# FIRE
%cd {REPO_CODE_DIR}/module1
!python apply_to_progression_data.py \
    --images-dir /content/data/FIRE_dataset/FIRE/Images \
    --classifier-checkpoint "{CLS_CKPT}"{CLS_NORM_ARGS} \
    --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
    --checkpoint-every 50 --resume \
    --out "{MODULE1_CACHE_DIR}/module1_outputs_fire.pt"


/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc/module1
Classifier normalisation: fgadr mean=[0.4577976167201996, 0.2596849203109741, 0.13276457786560059] std=[0.28394654393196106, 0.18262696266174316, 0.13211798667907715]
Lesion segmentation: tiled (retina 1280 px, 512 px tiles, stride 384) | retina crop: on
/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc/module1/apply_to_progression_data.py:190: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return np.array(Image.fromarray(soft.astype(np.float32), mode="F").resize(size, Image.BILINEAR))
✓ [1/268] A01_1: grade=4 lbs=0.01207 (lesions: ['EX', 'HE', 'MA', 'SE'])
✓ [2/268] A01_2: grade=3 lbs=0.00524 (lesions: ['EX', 'HE', 'MA', 'SE'])
✓ [3/268] A02_1: grade=4 lbs=0.00150 (lesions: ['EX', 'HE', 'MA', 'SE'])
✓ [4/268] A02_2: grade=4 lbs=0.04471 (lesions: ['EX', 'HE', 'MA', 'SE'])
✓ [5/268] A03_1: grade=4 lbs=0.00150 (lesions: ['EX', 'HE', 'MA', 'SE'])
✓ [6/268

In [9]:
# LongDR
!python apply_to_progression_data.py \
    --images-dir /content/data/LongDRScreening_20150209/FundusImagesNormalized \
    --classifier-checkpoint "{CLS_CKPT}"{CLS_NORM_ARGS} \
    --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
    --checkpoint-every 50 --resume \
    --out "{MODULE1_CACHE_DIR}/module1_outputs_longdr.pt"


Classifier normalisation: fgadr mean=[0.4577976167201996, 0.2596849203109741, 0.13276457786560059] std=[0.28394654393196106, 0.18262696266174316, 0.13211798667907715]
Lesion segmentation: tiled (retina 1280 px, 512 px tiles, stride 384) | retina crop: on
/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc/module1/apply_to_progression_data.py:190: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return np.array(Image.fromarray(soft.astype(np.float32), mode="F").resize(size, Image.BILINEAR))
✓ [1/1120] eye_001__visit_1_image_1: grade=3 lbs=0.07298 (lesions: ['EX', 'HE', 'MA', 'SE'])
✓ [2/1120] eye_001__visit_1_image_2: grade=3 lbs=0.09861 (lesions: ['EX', 'HE', 'MA', 'SE'])
✓ [3/1120] eye_001__visit_1_image_3: grade=3 lbs=0.09673 (lesions: ['EX', 'HE', 'MA', 'SE'])
✓ [4/1120] eye_001__visit_1_image_4: grade=3 lbs=0.07632 (lesions: ['EX', 'HE', 'MA', 'SE'])
✓ [5/1120] eye_001__visit_2_image_1: grade=3 lbs=0.08210 (lesions: ['EX'

In [10]:
# Tianjin -- the slow one. If this gets interrupted, just re-run this exact cell: --resume
# means it picks up where it left off instead of starting over.
!python apply_to_progression_data.py \
    --images-dir "{TIANJIN_DIR}/baseline fundus images" \
    --classifier-checkpoint "{CLS_CKPT}"{CLS_NORM_ARGS} \
    --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
    --checkpoint-every 50 --resume \
    --out "{MODULE1_CACHE_DIR}/module1_outputs_tianjin.pt"


Classifier normalisation: fgadr mean=[0.4577976167201996, 0.2596849203109741, 0.13276457786560059] std=[0.28394654393196106, 0.18262696266174316, 0.13211798667907715]
Lesion segmentation: tiled (retina 1280 px, 512 px tiles, stride 384) | retina crop: on
⚠ 00194__._00194-7254: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7254.jpg')
⚠ 00194__._00194-7256: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7256.jpg')
⚠ 00194__._00194-7258: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7258.jpg')
⚠ 00194__._00194-7265: failed (cannot identify image file '/content/data/retinal-dr-longitudinal/baseline fundus images/00194/._00194-7265.jpg')
/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc/module1/apply_to_progression_data.py:190: DeprecationWarning: 'mode' parameter is deprecated and will 

## Done

All 3 caches are now on Drive under `module1_cache_drgnet_comb/` (`MODULE1_SOURCE='drgnet_combined'`),
`module1_cache_drgnet/` (`'drgnet'`) or `module1_cache/` (`'poc'`). Next: notebook 09 (EyePACS pretraining), then 06 with the same
`MODULE1_SOURCE`, then 07, 08, 10.

## Before closing the runtime

Files written to Drive upload in the background. Run this last cell and wait for the ✓ before
disconnecting or deleting the runtime -- otherwise large files written at the end (model weights,
caches) can be lost.

In [11]:
from google.colab import drive
drive.flush_and_unmount()
print('✓ Everything is uploaded to Google Drive -- safe to disconnect the runtime.')

✓ Everything is uploaded to Google Drive -- safe to disconnect the runtime.
